# QO-BRA: Quantum Operator-Based Real-Amplitude Autoencoder

A quantum machine learning framework for generating biologically plausible protein sequences.
QO-BRA combines variational quantum circuits with ESM (Evolutionary Scale Modeling) loss to
generate novel protein sequences that match the statistical properties of natural proteins.

### Architecture

```
Input Sequence -> [Encoder] -> Latent State -> [Decoder] -> Output Sequence
     |                            |                          |
  Feature Map           Gaussian-like space           Reconstruction
  (RawFeatureVector)    (trained via MMD)             (trained via Fidelity + ESM)
```

### Two-Phase Training

| Phase | Loss Function | What is Trained | Goal |
|-------|---------------|-----------------|------|
| **Phase 1** | MMD (Maximum Mean Discrepancy) | Encoder | Learn latent space matching Gaussian distribution |
| **Phase 2** | Fidelity + ESM | Decoder (encoder frozen) | Reconstruct biologically plausible sequences |

### Notebook Outline

0. Environment Setup
1. Configuration
2. Module Initialization & Circuit Visualization
3. Data Exploration
4. Phase 1: Encoder Training (MMD)
5. Phase 2: Decoder Training (Fidelity + ESM)
6. Evaluation
7. De Novo Sequence Generation
8. ESM Evaluation (Optional)

---
## 0. Environment Setup

This notebook runs on **Google Colab** or any Jupyter environment.

The cell below will:
1. Install all required Python packages
2. Clone the repository (on Colab) or verify local files exist
3. Change directory to `src/` where the source code lives

If running **locally**, make sure this notebook is inside the `src/` directory,
or adjust the path in the cell below.

In [ ]:
# Install dependencies
!pip install -q qiskit==1.4.2 qiskit-algorithms==0.3.1 qiskit-machine-learning==0.8.2
!pip install -q torch numpy scipy matplotlib tqdm
!pip install -q fair-esm

import os

# --- Colab: clone the repo and navigate into src/ ---
# If running on Colab, uncomment the two lines below and paste your repo URL:
# !git clone https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git
# os.chdir('<YOUR_REPO>/src')

# --- Local: ensure we are in the src/ directory ---
if os.path.basename(os.getcwd()) != 'src' and os.path.isdir('src'):
    os.chdir('src')

# Verify training data is accessible
assert os.path.isdir('Training data'), (
    f"'Training data/' not found in {os.getcwd()}. "
    "Make sure this notebook runs from the src/ directory."
)
print(f"Working directory: {os.getcwd()}")
print(f"Training data found: {os.listdir('Training data/')[:6]}...")

---
## 1. Configuration

Edit the parameters below before running. These control the quantum circuit architecture,
loss function weights, and ESM settings.

| Parameter | Description | Default |
|-----------|-------------|---------|
| `METALS` | Metal types to train on (e.g. `["Zn"]`, `["Ca", "Mg"]`) | `["Zn"]` |
| `NUM_QUBITS` | Total qubits in the circuit (sequence capacity = 2^N - 1) | `9` |
| `REPS` | Ansatz layers (more = more expressive, slower) | `2` |
| `LAMBDA_MMD` | MMD loss weight (Phase 1) | `1.0` |
| `LAMBDA_FIDELITY` | Fidelity loss weight (Phase 2) | `1.0` |
| `LAMBDA_ESM` | ESM loss weight (Phase 2, 0 to disable) | `0.001` |
| `WARMUP_ESM` | Iterations to linearly ramp ESM lambda (None = no warmup) | `500` |
| `ESM_K` | Positions masked per sequence for ESM scoring | `32` |
| `ESM_MODEL` | ESM model name (smaller = faster training) | `esm2_t6_8M_UR50D` |

In [ ]:
# ======================== USER PARAMETERS ========================
# Edit these values, then run all cells below.

METALS       = ["Zn"]                  # Metal-binding protein types
NUM_QUBITS   = 9                       # Qubits (sequence capacity = 2^N - 1 = 511 residues)
REPS         = 2                       # Ansatz repetitions / layers

LAMBDA_MMD     = 1.0                   # Phase 1: MMD weight
LAMBDA_FIDELITY = 1.0                  # Phase 2: Fidelity weight
LAMBDA_ESM     = 0.001                 # Phase 2: ESM weight (0 = disable ESM loss)
WARMUP_ESM     = 500                   # ESM warmup iterations (None = no warmup)
ESM_K          = 32                    # Masked positions per sequence for ESM
ESM_MODEL      = "esm2_t6_8M_UR50D"   # ESM model (use small model for training speed)

# =================================================================

---
## 2. Module Initialization

This cell feeds the parameters above into the QO-BRA module chain via `sys.argv`,
then imports everything. The import triggers the full initialization pipeline:

1. **ansatz.py** parses arguments, builds the RealAmplitudes quantum circuit ansatz
2. **coding.py** defines amino acid encoding/decoding and novelty checking
3. **model.py** constructs encoder (`qc_e`), decoder (`qc_d`), and autoencoder (`qc_ed`) circuits
4. **inputs.py** loads protein sequences, creates 80/20 train/test splits, draws circuit diagrams
5. **count.py** builds token-to-amplitude mappings, initializes circuit parameters
6. **config.py** sets training hyperparameters (batch size, frequencies, epochs)
7. **cost.py** assembles the two-phase loss functions

> This cell takes a moment to run as it loads and preprocesses all training data.

In [ ]:
%matplotlib inline
import sys, os

# Build sys.argv so argparse in ansatz.py sees the right parameters
argv = ["notebook"] + METALS
argv += ["--num-qubits", str(NUM_QUBITS)]
argv += ["--reps", str(REPS)]
argv += ["--lambda-mmd", str(LAMBDA_MMD)]
argv += ["--lambda-fidelity", str(LAMBDA_FIDELITY)]
argv += ["--lambda-esm", str(LAMBDA_ESM)]
argv += ["--esm-k", str(ESM_K)]
argv += ["--esm-model", ESM_MODEL]
if WARMUP_ESM is not None:
    argv += ["--warmup-esm", str(WARMUP_ESM)]
argv += ["--mode", "0"]  # training mode

sys.argv = argv
print(f"sys.argv = {sys.argv}")

# Import triggers the full initialization chain
from cost import *
from scipy.optimize import minimize
from qiskit_algorithms.optimizers import COBYLA

print("\n" + "=" * 60)
print("QO-BRA initialization complete")
print("=" * 60)
print(f"Training samples : {len(train_seqs)}")
print(f"Test samples     : {len(test_seqs)}")
print(f"Qubits           : {num_tot}")
print(f"State dimension  : {dim_tot}")
print(f"Encoder params   : {num_encode}")
print(f"Decoder params   : {num_decode}")
print(f"Experiment tag   : {S}")
print("=" * 60)

### Quantum Circuit Diagrams

The three circuits created during initialization:

In [ ]:
from qiskit.visualization import circuit_drawer
from IPython.display import display, Image

print("Encoder circuit (qc_e):")
display(circuit_drawer(qc_e, output='mpl', style='clifford'))

print("\nFull autoencoder circuit (qc_ed):")
display(circuit_drawer(qc_ed, output='mpl', style='clifford'))

print("\nDecoder circuit (qc_d):")
display(circuit_drawer(qc_d, output='mpl', style='clifford'))

---
## 3. Data Exploration

### Sequence Format

Sequences use a specialized annotation format:

| Symbol | Meaning |
|--------|---------|
| `A-Y` | Standard amino acids |
| `+` | Ligand-binding residue marker (follows the amino acid) |
| `:` | Chain boundary separator |
| `X` | Sequence terminator |

**Example:** `MKFL+VLC+GKDF:PEPTIDE+CHAIN:` means L and C are binding residues, and there are two chains separated by `:`.

In [ ]:
print("=" * 60)
print("Sample Training Sequences")
print("=" * 60)
for i, seq in enumerate(train_seqs[:5]):
    idx = seq.find("X")
    display_seq = seq[:idx] if idx != -1 else seq
    n_plus = display_seq.count("+")
    n_colon = display_seq.count(":")
    raw_len = len("".join(c for c in display_seq if c not in "+X"))
    print(f"\n[{i}] length={raw_len}  binding_sites={n_plus}  chains={n_colon + 1}")
    print(f"    {display_seq[:120]}{'...' if len(display_seq) > 120 else ''}")

### Token Frequency Distribution (Gaussian Order)

The token-to-amplitude mapping arranges amino acid frequencies in Gaussian order
to optimize quantum state preparation. The bar chart below shows this mapping.

In [ ]:
import matplotlib.pyplot as plt

cnt_x3 = list(gauss_sort.keys())
cnt_y3 = [cnt[x] for x in cnt_x3]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(keys, cnt_y3, width=1.8, color='forestgreen', edgecolor='white', linewidth=0.3)
for j, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{keys[j]:.1f}', ha='center', va='bottom', fontsize=4.5)
ax.set_title('Token frequencies in Gaussian amplitude order', fontsize=14)
ax.set_xticks(keys, cnt_x3, fontsize=7)
ax.set_xlabel('Token (amplitude value)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

### Training Data Distributions

Distributions of sequence length, number of chains, and binding site counts across the training set.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(len_cnts_real, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Sequence Length Distribution')
axes[0].set_xlabel('Length (residues)')
axes[0].set_ylabel('Count')

axes[1].hist(dn_cnts_real, bins=range(max(dn_cnts_real)+2), color='coral', edgecolor='white', align='left')
axes[1].set_title('Chain Count Distribution')
axes[1].set_xlabel('Number of chains')
axes[1].set_ylabel('Count')

axes[2].hist(plus_cnts_real, bins=30, color='mediumpurple', edgecolor='white')
axes[2].set_title('Binding Site Distribution')
axes[2].set_xlabel('Number of binding sites')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Sequence length  : min={min(len_cnts_real)}, max={max(len_cnts_real)}, median={sorted(len_cnts_real)[len(len_cnts_real)//2]}")
print(f"Chains per seq   : min={min(dn_cnts_real)}, max={max(dn_cnts_real)}")
print(f"Binding sites    : min={min(plus_cnts_real)}, max={max(plus_cnts_real)}, median={sorted(plus_cnts_real)[len(plus_cnts_real)//2]}")

---
## 4. Phase 1: Encoder Training (MMD Loss)

**Objective:** Learn an encoder that maps protein sequences to a well-structured latent space
(Gaussian distribution).

$$\mathcal{L}_{\text{Phase 1}} = \lambda_{\text{MMD}} \times L_{\text{MMD}}$$

- Only encoder parameters are optimized
- MMD (Maximum Mean Discrepancy) with RBF kernel matches encoded distribution to target Gaussian
- Optimizer: COBYLA (gradient-free, suitable for quantum circuits)

In [ ]:
# Pre-compute encoded states (one-time cost)
precompute_encoded_states(train_seqs, test_seqs)

# Initialize mini-batch iterator
if MINI_BATCH_ENABLED:
    init_batch_iterator(len(train_seqs), batch_size=MINI_BATCH_SIZE, seed=42)

# Cache target distribution
print("Pre-generating target distribution...")
_ = get_cached_target(len(train_seqs), mu, std)
print("Target distribution cached.")

print(f"\nPhase 1 configuration:")
print(f"  Loss       = {LAMBDA_MMD} * L_MMD")
print(f"  Parameters = {num_encode} (encoder)")
print(f"  Max epochs = {MAX_EPOCHS_ENCODER}")

In [ ]:
import time as _time

start_phase1 = _time.time()

opt_encoder = COBYLA(maxiter=MAX_EPOCHS_ENCODER)
pbar = init_progress_bar(max_epochs=MAX_EPOCHS_ENCODER)

f_encoder = partial(encoder_loss, train_input=train_seqs, test_input=test_seqs)

try:
    encoder_result = opt_encoder.minimize(fun=f_encoder, x0=xe)
finally:
    close_progress_bar()

xe_opt = encoder_result.x

with open(f'{S}/opt-e-{S}.pkl', 'wb') as F:
    pickle.dump(xe_opt, F)

elapsed_phase1 = (_time.time() - start_phase1) / 3600
print(f"\nPhase 1 completed in {elapsed_phase1:.2f} h")
print(f"Saved encoder parameters to {S}/opt-e-{S}.pkl")

### Phase 1 Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(trains_k, 'b-', label='Train MMD', alpha=0.8)
ax.plot(tests_k, 'b:', label='Test MMD', alpha=0.8)
ax.set_title('Phase 1: Encoder MMD Loss')
ax.set_xlabel('Iteration')
ax.set_ylabel('MMD Loss')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Phase 2: Decoder Training (Fidelity + ESM Loss)

**Objective:** Learn a decoder that reconstructs biologically plausible sequences from
latent representations.

$$\mathcal{L}_{\text{Phase 2}} = \lambda_{\text{fidelity}} \times L_{\text{Fidelity}} + \lambda_{\text{ESM}} \times L_{\text{ESM}}$$

- Encoder is **frozen** (uses optimized parameters from Phase 1)
- Only decoder parameters are optimized
- Fidelity loss ensures accurate reconstruction ($F = |\langle\psi|\phi\rangle|^2$)
- ESM loss ensures biological plausibility (pseudo-log-likelihood)
- Optional linear warmup for ESM lambda to stabilize early training

In [ ]:
start_phase2 = _time.time()

# Freeze encoder with optimized parameters
set_frozen_encoder(xe_opt)

# Reset batch iterator for phase 2
if MINI_BATCH_ENABLED:
    init_batch_iterator(len(train_seqs), batch_size=MINI_BATCH_SIZE, seed=43)

reset_decoder_tracking()

opt_decoder = COBYLA(maxiter=MAX_EPOCHS_DECODER)
pbar = init_progress_bar(max_epochs=MAX_EPOCHS_DECODER)

f_decoder = partial(decoder_loss, train_input=train_seqs, test_input=test_seqs)

try:
    decoder_result = opt_decoder.minimize(fun=f_decoder, x0=xd)
finally:
    close_progress_bar()

xd_opt = decoder_result.x

with open(f'{S}/opt-d-{S}.pkl', 'wb') as F:
    pickle.dump(xd_opt, F)

elapsed_phase2 = (_time.time() - start_phase2) / 3600
print(f"\nPhase 2 completed in {elapsed_phase2:.2f} h")
print(f"Saved decoder parameters to {S}/opt-d-{S}.pkl")

# Update global parameters
xe = xe_opt
xd = xd_opt

### Phase 2 Loss Curves

In [ ]:
n_plots = 1 + (1 if LAMBDA_ESM > 0 and len(trains_esm) > 0 else 0)
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

axes[0].plot(trains_fidelity, 'g-', label='Train Fidelity', alpha=0.8)
axes[0].plot(tests_fidelity, 'g:', label='Test Fidelity', alpha=0.8)
axes[0].set_title('Phase 2: Fidelity Loss (Decoder)')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Infidelity (1 - F)')
axes[0].legend()

if n_plots > 1:
    axes[1].plot(trains_esm, 'r-', label='Train ESM', alpha=0.8)
    axes[1].plot(tests_esm, 'r:', label='Test ESM', alpha=0.8)
    axes[1].set_title('Phase 2: ESM Loss (Decoder)')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Negative PLL')
    axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6. Evaluation

Evaluate reconstruction accuracy on both the training and test sets.
The `output()` function runs the full autoencoder (encode then decode) on every sequence
and measures the fraction of perfectly reconstructed sequences.

In [ ]:
start_eval = _time.time()

# Initialize results files
with open(f"{S}/Results-{S}.txt", "w") as f:
    f.write("Dataset\tSize\tR\n")
with open(f"{S}/R-{S}.txt", "w") as f:
    f.write("TRAINING SET\n")

print("Evaluating training set...")
output(train_seqs, head, "Train", encoder_params=xe, decoder_params=xd)

with open(f"{S}/R-{S}.txt", "a") as f:
    f.write("TEST SET\n")

print("\nEvaluating test set...")
output(test_seqs, head, "Test", encoder_params=xe, decoder_params=xd)

elapsed_eval = (_time.time() - start_eval) / 60
print(f"\nEvaluation completed in {elapsed_eval:.2f} min")
print(f"\nTotal training time:")
print(f"  Phase 1 (Encoder): {elapsed_phase1:.2f} h")
print(f"  Phase 2 (Decoder): {elapsed_phase2:.2f} h")
print(f"  Evaluation       : {elapsed_eval:.2f} min")

### View Results

Display the reconstruction accuracy summary and a few sample input/output pairs.

In [ ]:
print("=" * 60)
print("Reconstruction Accuracy")
print("=" * 60)
with open(f"{S}/Results-{S}.txt", "r") as f:
    print(f.read())

print("\n" + "=" * 60)
print("Sample Reconstructions (first 5)")
print("=" * 60)
with open(f"{S}/R-{S}.txt", "r") as f:
    lines = f.readlines()
count = 0
for line in lines:
    print(line.rstrip())
    if line.startswith("*"):
        count += 1
        if count >= 5:
            print("... (truncated)")
            break

---
## 7. De Novo Sequence Generation

Generate novel protein sequences by sampling from the learned latent distribution
and decoding through the trained decoder. Each generated sequence is checked for:

- **Novelty (N):** Not present in the training set (fast k-mer + LCS check)
- **Uniqueness (U):** Not a duplicate of another generated sequence
- **Validity (V):** Meets biological constraints (has binding sites, valid chain structure, minimum length)

Only sequences that are novel AND unique AND valid are saved.

> **Note:** By default this runs 20 seeds. Set `N_SEEDS` below to control how many.

In [ ]:
N_SEEDS = 1  # Number of generation seeds to run (default in gen.py is 20)

In [ ]:
import re as Re
from gen_func import *

Nlist, Ulist, Vlist = [], [], []
begin = _time.time()

pattern3 = r".\+.\+.\+"
pattern5 = r".\+.\+.\+.\+.\+"

# Build k-mer index for fast novelty checking
print("Building k-mer index for fast novelty checking...")
kmer_index, seq_kmers = build_kmer_index(seqs, k=4)
print(f"Index built: {len(kmer_index)} unique 4-mers from {len(seqs)} training sequences")

all_valid_sequences = []  # Collect for ESM evaluation later

for seed in range(N_SEEDS):
    print(f"\n{'=' * 60}")
    print(f"Generation seed {seed}")
    print(f"{'=' * 60}")

    filename = f"{S}/NUV-{S}-{seed}.txt"
    with open(filename, "w") as f:
        f.write("N\tU\tV\n")

    torch.manual_seed(seed)
    np.random.seed(seed)

    N_count, U_count, V_count = 0, 0, 0
    valid = []
    start = _time.time()

    n = int(train_size * 2)
    denovos = make_target(n, mu, std)

    folder = f"{S}/{seed}"
    os.makedirs(folder, exist_ok=True)

    gen = generate_batch(denovos)

    n = len(gen)
    seen_sequences = set()
    log_interval = max(1, n // 10)
    last_log_time = _time.time()

    for i in range(len(gen)):
        s = gen[i]
        idx = s.find("X")
        s2 = s[:idx] if idx != -1 else s

        novel = check_novelty_fast(s2, seqs, kmer_index, seq_kmers, k=4)
        if novel:
            N_count += 1

        unique = s2 not in seen_sequences
        if unique:
            U_count += 1
            seen_sequences.add(s2)

        match3 = Re.findall(pattern3, s2)
        match5 = Re.findall(pattern5, s2)
        is_valid = False
        s4 = None

        if "+" in s2 and '::' not in s2 and len(match5) == 0 and len(match3) < 2:
            s3 = ''.join([c for c in s2 if c != "+"])
            s4 = s2.split(':')
            s5 = s3.split(':')
            if s5[-1] == '':
                s5 = s5[:-1]
            if all(len(l) >= threshold for l in s5) or \
               (':' not in s3 and len(s3) >= threshold):
                is_valid = True
                V_count += 1

        if novel and unique and is_valid:
            valid.append(s)
            HL = {}
            for j in range(len(s4)):
                chain, chainID = s4[j], alphabets[j]
                shift = 0
                for k in range(len(chain)):
                    if chain[k] == "+":
                        if chainID in HL:
                            HL[chainID].append(k - shift)
                        else:
                            HL[chainID] = [k - shift]
                        shift += 1
            prot_folder = f'{folder}/Samples/{i}'
            pml(s2, HL, i, prot_folder)

        if (i + 1) % log_interval == 0 or i == len(gen) - 1:
            elapsed = _time.time() - last_log_time
            rate = log_interval / elapsed if elapsed > 0 else 0
            print(f"  {i+1}/{len(gen)} | Valid: {len(valid)}/{train_size} | "
                  f"N:{N_count} U:{U_count} V:{V_count} | {rate:.1f} seq/s")
            last_log_time = _time.time()

        if len(valid) == train_size:
            n = i
            break

    size = min(n, len(gen))

    with open(f"{folder}/denovo-{seed}.txt", "w") as f:
        f.write("De novo sequences\n")

    cnt_gen, dn_cnts_gen, len_cnts_gen, plus_cnts_gen = cnts(valid, folder, seed)
    all_valid_sequences.extend(valid)

    elapsed = (_time.time() - start) / 3600
    print(f"  Generated in {elapsed:.2f} h")

    Nlist.append(N_count / size)
    Ulist.append(U_count / size)
    Vlist.append(V_count / size)

    with open(filename, "a") as f:
        f.write(f"{np.mean(Nlist):.3f}\t{np.mean(Ulist):.3f}\t{np.mean(Vlist):.3f}\n")
        f.write(f"{np.std(Nlist):.3f}\t{np.std(Ulist):.3f}\t{np.std(Vlist):.3f}\n")

    # --- Amino acid frequency comparison ---
    cnt_y2 = [cnt_gen[x] if x in cnt_gen else 0 for x in cnt_x]
    cnt_x2 = []
    for ii in range(len(cnt_x)):
        cnt_x2.append(cnt_x[ii][0] + "\n+" if cnt_x[ii][-1] == "+" else cnt_x[ii])

    cnt_y1_arr = np.array(cnt_y1)
    cnt_y2_arr = np.array(cnt_y2)

    x_pos = np.arange(len(cnt_x2))
    width = 0.36

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(x_pos - width/2, cnt_y1_arr, width, label='Training')
    ax.bar(x_pos + width/2, cnt_y2_arr, width, label='De novo')
    ax.set_title(f'Amino Acid Frequency Comparison (seed={seed})')
    ax_inset = inset_axes(ax, width="30%", height="50%", loc='right')
    ax_inset.bar(x_pos - width/2, cnt_y1_arr, width)
    ax_inset.bar(x_pos + width/2, cnt_y2_arr, width)
    ax_inset.xaxis.set_visible(False)
    ax_inset.set_yscale('log')
    ax_inset.tick_params(axis='y', labelsize=8)
    ax.set_xticks(x_pos, cnt_x2)
    ax.set_xlabel('AA / AA+')
    ax.set_ylabel('Frequency')
    ax.legend()
    fig.savefig(f"{folder}/Bar-{seed}-{S}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # --- Property distribution comparisons ---
    Plot(dn_cnts_real, dn_cnts_gen, 'chain numbers', folder, seed)
    plt.show()
    Plot(plus_cnts_real, plus_cnts_gen, 'binding sites', folder, seed)
    plt.show()
    Plot(len_cnts_real, len_cnts_gen, 'length', folder, seed)
    plt.show()

total_elapsed = (_time.time() - begin) / 3600
print(f"\n{'=' * 60}")
print(f"Generation complete")
print(f"{'=' * 60}")
print(f"Total time: {total_elapsed:.2f} h")
print(f"Novelty  (N): {np.mean(Nlist):.3f} +/- {np.std(Nlist):.3f}")
print(f"Uniqueness(U): {np.mean(Ulist):.3f} +/- {np.std(Ulist):.3f}")
print(f"Validity (V): {np.mean(Vlist):.3f} +/- {np.std(Vlist):.3f}")
print(f"Total valid sequences generated: {len(all_valid_sequences)}")

### View Generated Sequences

In [ ]:
print(f"Showing first 10 of {len(all_valid_sequences)} generated sequences:\n")
for i, seq in enumerate(all_valid_sequences[:10]):
    idx = seq.find("X")
    display_seq = seq[:idx] if idx != -1 else seq
    raw_len = len("".join(c for c in display_seq if c not in "+X"))
    n_plus = display_seq.count("+")
    n_colon = display_seq.count(":")
    print(f"[{i}] len={raw_len}  binding={n_plus}  chains={n_colon+1}")
    print(f"    {display_seq[:120]}{'...' if len(display_seq) > 120 else ''}")
    print()

---
## 8. ESM Evaluation (Optional)

Score generated sequences with a larger ESM model to assess biological plausibility.
This section uses the `esm_eval` subpackage to:

1. Compute pseudo-log-likelihood scores for training and generated sequences
2. Compare score distributions with histograms
3. Compute statistical distances (KS statistic, Wasserstein distance)

| Range | Interpretation |
|-------|----------------|
| -1.5 to -0.5 | Natural proteins (high quality) |
| -2.5 to -1.0 | Good generated sequences |
| -4.0 to -2.5 | Poor/random sequences |

> **Note:** This uses a larger ESM model for evaluation accuracy. Set `EVAL_ESM_MODEL` below.

In [ ]:
EVAL_ESM_MODEL = "esm2_t33_650M_UR50D"  # Larger model for more accurate evaluation
EVAL_BATCH_SIZE = 8                       # Reduce if running out of GPU memory

In [ ]:
import sys as _sys
_sys.path.insert(0, '../esm_eval') if '../esm_eval' not in _sys.path else None

try:
    from esm_eval import ESMScorer, plot_log_likelihood_histograms
    from esm_eval import compute_pairwise_distances, load_distributions_from_csv
    from esm_loss import clean_sequence_for_esm
    import csv
    from pathlib import Path

    # Prepare sequences: clean QO-BRA format for ESM
    train_clean = [clean_sequence_for_esm(s) for s in train_seqs[:200]]
    gen_clean = [clean_sequence_for_esm(s) for s in all_valid_sequences[:200]]
    train_clean = [s for s in train_clean if len(s) > 0]
    gen_clean = [s for s in gen_clean if len(s) > 0]

    print(f"Scoring {len(train_clean)} training + {len(gen_clean)} generated sequences")
    print(f"Model: {EVAL_ESM_MODEL}")

    # Build combined data
    all_seqs = train_clean + gen_clean
    all_ids = [f"train_{i}" for i in range(len(train_clean))] + \
              [f"gen_{i}" for i in range(len(gen_clean))]
    all_sources = ["training"] * len(train_clean) + ["generated"] * len(gen_clean)

    # Score
    scorer = ESMScorer(model_name=EVAL_ESM_MODEL)
    scores = scorer.compute_log_likelihood_batch(
        sequences=all_seqs, sequence_ids=all_ids,
        source_files=all_sources, show_progress=True,
        batch_size=EVAL_BATCH_SIZE
    )
    results = scorer.scores_to_dict_list(scores)

    # Save CSV
    csv_path = f"{S}/esm_eval_results.csv"
    fieldnames = ["sequence_id", "source_file", "sequence", "length",
                  "total_log_likelihood", "mean_log_likelihood"]
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"\nResults saved to {csv_path}")

    # Histogram
    fig_path = f"{S}/esm_eval_histogram.png"
    plot_log_likelihood_histograms(results, output_path=fig_path)
    display(Image(filename=fig_path))

    # Summary statistics
    train_lls = [r["mean_log_likelihood"] for r in results if r["source_file"] == "training"]
    gen_lls = [r["mean_log_likelihood"] for r in results if r["source_file"] == "generated"]
    print(f"\nTraining  mean LL: {np.mean(train_lls):.4f} +/- {np.std(train_lls):.4f}")
    print(f"Generated mean LL: {np.mean(gen_lls):.4f} +/- {np.std(gen_lls):.4f}")

    # Distribution distances
    from scipy.stats import ks_2samp, wasserstein_distance
    ks_stat, ks_pval = ks_2samp(train_lls, gen_lls)
    w_dist = wasserstein_distance(train_lls, gen_lls)
    print(f"\nKS statistic : {ks_stat:.4f} (p={ks_pval:.4e})")
    print(f"Wasserstein  : {w_dist:.4f}")

except ImportError as e:
    print(f"ESM evaluation skipped (missing dependency): {e}")
    print("Install with: pip install fair-esm")
except Exception as e:
    print(f"ESM evaluation failed: {e}")
    import traceback
    traceback.print_exc()

---

## Summary

This notebook walked through the complete QO-BRA pipeline:

1. **Configuration** of quantum circuit parameters and loss weights
2. **Module initialization** including data loading and circuit construction
3. **Data exploration** of training protein sequences
4. **Phase 1** encoder training to learn a Gaussian latent space (MMD loss)
5. **Phase 2** decoder training to reconstruct biologically plausible sequences (Fidelity + ESM loss)
6. **Evaluation** of reconstruction accuracy on train/test sets
7. **De novo generation** of novel protein sequences with NUV quality metrics
8. **ESM evaluation** of generated sequence plausibility

All output files (parameters, results, plots, generated sequences) are saved under
the `Training data/{experiment_tag}/` directory.